# 01 · Carga y auditoría

**Objetivo:** unir los 7 CSV y revisar estructura, tipos, faltantes, duplicados, categorías y unidades
**Preguntas que responde:** ¿los datos son consistentes entre años? ¿Qué preguntas de investigación son viables?
**Entrada:** `data/raw/precio_mayorista_fruta-hortaliza_*.csv`
**Salida:** ninguna (solo diagnóstico; la limpieza ocurre en el notebook 02)
**Decisiones tomadas:** se registran en `docs/decisiones.md`

In [21]:
import sys
sys.path.append("..")

import pandas as pd
from src.carga import listar_archivos, leer_crudo, COLUMNAS_ESPERADAS

import numpy as np

RAW = "../data/raw"
FILAS_README = {2020: 192883, 2021: 194776, 2022: 188313, 2023: 188192,
                2024: 189754, 2025: 202029, 2026: 141588}

archivos = listar_archivos(RAW)
print(len(archivos), "archivos encontrados")

dfs = {ruta.name: leer_crudo(ruta) for ruta in archivos}

resumen = pd.DataFrame([
    {
        "archivo": nombre,
        "anio": int(nombre[-8:-4]),
        "filas": len(d),
        "columnas": d.shape[1],
        "encabezado_ok": list(d.columns) == COLUMNAS_ESPERADAS,
    }
    for nombre, d in dfs.items()
])
resumen["filas_readme"] = resumen["anio"].map(FILAS_README)
resumen["coincide_readme"] = resumen["filas"] == resumen["filas_readme"]
print(resumen.to_string(index=False))
print("Total de filas:", resumen["filas"].sum())

# Si algún encabezado no coincide, mostrar la diferencia
for nombre, d in dfs.items():
    if list(d.columns) != COLUMNAS_ESPERADAS:
        print("\nEncabezado distinto en", nombre)
        print("  sobran:", set(d.columns) - set(COLUMNAS_ESPERADAS))
        print("  faltan:", set(COLUMNAS_ESPERADAS) - set(d.columns))

7 archivos encontrados
                                  archivo  anio  filas  columnas  encabezado_ok  filas_readme  coincide_readme
precio_mayorista_fruta-hortaliza_2020.csv  2020 192883        14           True        192883             True
precio_mayorista_fruta-hortaliza_2021.csv  2021 194776        14           True        194776             True
precio_mayorista_fruta-hortaliza_2022.csv  2022 188313        14           True        188313             True
precio_mayorista_fruta-hortaliza_2023.csv  2023 188192        14           True        188192             True
precio_mayorista_fruta-hortaliza_2024.csv  2024 189754        14           True        189754             True
precio_mayorista_fruta-hortaliza_2025.csv  2025 202029        14           True        202029             True
precio_mayorista_fruta-hortaliza_2026.csv  2026 141588        14           True        141588             True
Total de filas: 1297535


In [2]:
df = pd.concat(
    [d.assign(archivo=nombre) for nombre, d in dfs.items()],
    ignore_index=True,
)
del dfs  # libera memoria

print(df.shape)
print(df.head(3).T)   # transpuesta para leer cómodo las 14 columnas

(1297535, 15)
                                                                    0  \
Fecha                                                      2020-01-02   
ID region                                                           5   
Region                                           Región de Valparaíso   
Mercado                                          Femacal de La Calera   
Subsector                                     Hortalizas y tubérculos   
Producto                                                       Acelga   
Variedad / Tipo                                       Sin especificar   
Calidad                                                       Primera   
Unidad de comercializacion               $/docena de atados (6 kilos)   
Origen                                          Provincia de Quillota   
Volumen                                                           120   
Precio minimo                                               1800,0000   
Precio maximo                        

In [3]:
# --- Fechas ---
fecha = pd.to_datetime(df["Fecha"], format="%Y-%m-%d", errors="coerce")
print("Fechas no interpretables:", fecha.isna().sum())

df["anio_archivo"] = df["archivo"].str[-8:-4].astype(int)
df["anio_fecha"] = fecha.dt.year

print("\n¿Cada archivo contiene solo fechas de su año?")
print(pd.crosstab(df["anio_archivo"], df["anio_fecha"]))

print("\nRango de fechas por archivo:")
print(fecha.groupby(df["anio_archivo"]).agg(["min", "max"]))

# --- Números ---
columnas_num = ["Volumen", "Precio minimo", "Precio maximo", "Precio promedio"]
print("\nColumnas numéricas:")
for c in columnas_num:
    limpio = df[c].str.replace(",", ".", regex=False).str.strip()
    num = pd.to_numeric(limpio, errors="coerce")
    vacios = (limpio == "").sum()
    no_numericos = num.isna().sum() - vacios
    print(f"{c:15s} vacíos={vacios:<7} no_numéricos={no_numericos:<7} "
          f"min={num.min():<12} max={num.max()}")

Fechas no interpretables: 0

¿Cada archivo contiene solo fechas de su año?
anio_fecha      2020    2021    2022    2023    2024    2025    2026
anio_archivo                                                        
2020          192883       0       0       0       0       0       0
2021               0  194776       0       0       0       0       0
2022               0       0  188313       0       0       0       0
2023               0       0       0  188192       0       0       0
2024               0       0       0       0  189754       0       0
2025               0       0       0       0       0  202029       0
2026               0       0       0       0       0       0  141588

Rango de fechas por archivo:
                    min        max
anio_archivo                      
2020         2020-01-02 2020-12-31
2021         2021-01-04 2021-12-31
2022         2022-01-03 2022-12-30
2023         2023-01-03 2023-12-29
2024         2024-01-02 2024-12-31
2025         2025-01-01 2025-

In [4]:
print(f"{df.memory_usage(deep=True).sum() / 1024**2:.0f} MB en memoria")
print(df["ID region"].nunique(), "valores distintos de ID region")

1247 MB en memoria
9 valores distintos de ID region


In [5]:
cols_cat = ["ID region", "Region", "Mercado", "Subsector", "Producto",
            "Variedad / Tipo", "Calidad", "Unidad de comercializacion",
            "Origen", "archivo"]
for c in cols_cat:
    df[c] = df[c].astype("category")

df["Fecha"] = fecha
for c in columnas_num:
    df[c] = pd.to_numeric(df[c].str.replace(",", ".", regex=False))

df["anio"] = df["Fecha"].dt.year
df = df.drop(columns=["anio_archivo", "anio_fecha"])

print(f"{df.memory_usage(deep=True).sum() / 1024**2:.0f} MB en memoria")
print(df.dtypes)

69 MB en memoria
Fecha                         datetime64[us]
ID region                           category
Region                              category
Mercado                             category
Subsector                           category
Producto                            category
Variedad / Tipo                     category
Calidad                             category
Unidad de comercializacion          category
Origen                              category
Volumen                                int64
Precio minimo                        float64
Precio maximo                        float64
Precio promedio                      float64
archivo                             category
anio                                   int32
dtype: object


In [6]:
for c in columnas_num:
    n = (df[c] == 0).sum()
    print(f"{c:15s} ceros: {n:>8}  ({n / len(df) * 100:.2f}%)")

print()
print(df[columnas_num].describe(percentiles=[.01, .5, .99, .999]).round(1))

print("\nPrecio promedio exactamente 999990:", (df["Precio promedio"] == 999990).sum())
print("\nPrecio promedio igual a 0 con volumen > 0:",
      ((df["Precio promedio"] == 0) & (df["Volumen"] > 0)).sum())
print("Volumen igual a 0 con precio > 0:",
      ((df["Volumen"] == 0) & (df["Precio promedio"] > 0)).sum())

cols_ver = ["Fecha", "Mercado", "Producto", "Unidad de comercializacion",
            "Volumen", "Precio minimo", "Precio maximo", "Precio promedio"]
print("\nPrecios más altos:")
print(df.nlargest(5, "Precio promedio")[cols_ver].to_string(index=False))
print("\nVolúmenes más altos:")
print(df.nlargest(5, "Volumen")[cols_ver].to_string(index=False))

Volumen         ceros:      201  (0.02%)
Precio minimo   ceros:      201  (0.02%)
Precio maximo   ceros:      201  (0.02%)
Precio promedio ceros:      201  (0.02%)

         Volumen  Precio minimo  Precio maximo  Precio promedio
count  1297535.0      1297535.0      1297535.0        1297535.0
mean      1148.4        24697.1        25306.0          25001.8
std       6692.6        60758.2        61750.0          61247.0
min          0.0            0.0            0.0              0.0
1%           8.0          200.0          200.0            200.0
50%        240.0        10000.0        10000.0          10000.0
99%      20000.0       330000.0       340000.0         335000.0
99.9%    90000.0       495000.0       500000.0         495000.0
max    1100000.0       999990.0       999990.0         999990.0

Precio promedio exactamente 999990: 1

Precio promedio igual a 0 con volumen > 0: 0
Volumen igual a 0 con precio > 0: 0

Precios más altos:
     Fecha                                   Mercado P

In [7]:
orden_ok = (df["Precio minimo"] <= df["Precio promedio"]) & (df["Precio promedio"] <= df["Precio maximo"])
print("Filas que violan mínimo ≤ promedio ≤ máximo:", (~orden_ok).sum())
print(df.loc[~orden_ok, cols_ver].head(5).to_string(index=False))

igual = df["Precio minimo"] == df["Precio maximo"]
print(f"\nFilas con mínimo = máximo: {igual.sum()} ({igual.mean() * 100:.1f}%)")
print("\n% por año:")
print((igual.groupby(df["anio"]).mean() * 100).round(1))

Filas que violan mínimo ≤ promedio ≤ máximo: 0
Empty DataFrame
Columns: [Fecha, Mercado, Producto, Unidad de comercializacion, Volumen, Precio minimo, Precio maximo, Precio promedio]
Index: []

Filas con mínimo = máximo: 718907 (55.4%)

% por año:
anio
2020    52.4
2021    49.5
2022    50.3
2023    56.8
2024    58.7
2025    59.9
2026    61.8
dtype: float64


In [8]:
print(df.groupby(["ID region", "Region"], observed=True).size())

ID region  Region                          
10         Región de Los Lagos                  73956
13         Región Metropolitana de Santiago    504306
15         Región de Arica y Parinacota         43758
16         Región de Ñuble                      73085
4          Región de Coquimbo                  158784
5          Región de Valparaíso                153591
7          Región del Maule                     82403
8          Región del Biobío                    62969
9          Región de La Araucanía              144683
dtype: int64


In [9]:
for c in ["Variedad / Tipo", "Calidad", "Origen", "Mercado", "Producto"]:
    cats = df[c].cat.categories
    print(f"\n{c}: {len(cats)} valores distintos")
    print("  vacíos (''):", (df[c] == "").sum())
    print("  etiquetas con espacios al borde:", (cats != cats.str.strip()).sum())
    print("  colisiones al pasar a minúsculas:",
          len(cats) - cats.str.strip().str.lower().nunique())

print()
print(df["Variedad / Tipo"].value_counts(normalize=True).head(5).round(3))
print(df["Calidad"].value_counts().head(10))
print(df["Subsector"].value_counts())


Variedad / Tipo: 335 valores distintos
  vacíos (''): 0
  etiquetas con espacios al borde: 0
  colisiones al pasar a minúsculas: 0

Calidad: 70 valores distintos
  vacíos (''): 0
  etiquetas con espacios al borde: 0
  colisiones al pasar a minúsculas: 0

Origen: 81 valores distintos
  vacíos (''): 0
  etiquetas con espacios al borde: 0
  colisiones al pasar a minúsculas: 0

Mercado: 12 valores distintos
  vacíos (''): 0
  etiquetas con espacios al borde: 0
  colisiones al pasar a minúsculas: 0

Producto: 82 valores distintos
  vacíos (''): 0
  etiquetas con espacios al borde: 0
  colisiones al pasar a minúsculas: 0

Variedad / Tipo
Sin especificar    0.361
Hass               0.046
Larga vida         0.045
Granny Smith       0.023
Caramelo           0.021
Name: proportion, dtype: float64
Calidad
Primera         633610
Segunda         247348
Especial         95804
Tercera          44404
1a (guarda)      28689
1a (cosecha)     28127
Extra            27444
1a amarillo      21038
1a nueva(

In [10]:
clave = ["Fecha", "Mercado", "Producto", "Variedad / Tipo", "Calidad",
         "Unidad de comercializacion", "Origen"]

exacto_extra = df.duplicated(subset=COLUMNAS_ESPERADAS).sum()
exacto_todas = df.duplicated(subset=COLUMNAS_ESPERADAS, keep=False)
clave_todas = df.duplicated(subset=clave, keep=False)

print("Filas sobrantes si se eliminan duplicados exactos:", exacto_extra)
print("Filas que comparten la clave:", clave_todas.sum())

conflictos = df[clave_todas & ~exacto_todas]
print("Misma clave pero valores distintos:", len(conflictos))
print(conflictos.sort_values(clave).head(6)[clave + ["Volumen", "Precio promedio"]].to_string(index=False))

Filas sobrantes si se eliminan duplicados exactos: 0
Filas que comparten la clave: 0
Misma clave pero valores distintos: 0
Empty DataFrame
Columns: [Fecha, Mercado, Producto, Variedad / Tipo, Calidad, Unidad de comercializacion, Origen, Volumen, Precio promedio]
Index: []


In [11]:
print("Productos:", df["Producto"].nunique(),
      "| Mercados:", df["Mercado"].nunique(),
      "| Unidades:", df["Unidad de comercializacion"].nunique(),
      "| Orígenes:", df["Origen"].nunique())

print("\nMercados por año (filas):")
print(pd.crosstab(df["Mercado"], df["anio"]))

print("\nProductos distintos por año:")
print(df.groupby("anio")["Producto"].nunique())

Productos: 82 | Mercados: 12 | Unidades: 191 | Orígenes: 81

Mercados por año (filas):
anio                                        2020   2021   2022   2023   2024  \
Mercado                                                                        
Agrícola del Norte S.A. de Arica            7059   6078   6430   6451   6421   
Comercializadora del Agro de Limarí         4191   4620   4334   4336   4352   
Femacal de La Calera                       24915  22927  23350  22007  21402   
Feria Lagunitas de Puerto Montt            10750  10802  11988  11366  10819   
Macroferia Regional de Talca               11767  12421  12371  12508  12448   
Mapocho venta directa de Santiago           1786   1555   1288   1243   1301   
Mercado Mayorista Lo Valledor de Santiago  51864  49860  45273  42413  46608   
Terminal Hortofrutícola Agro Chillán        8945   8354   8776  13145  11723   
Terminal La Palmera de La Serena           18730  20309  19667  18140  18186   
Vega Central Mapocho de Santiago 

In [12]:
unidad = df["Unidad de comercializacion"].astype(str)
marca = unidad.str.contains("volumen", case=False)
print("Filas cuya unidad menciona 'volumen':", marca.sum())
print(unidad[marca].value_counts().to_string())

Filas cuya unidad menciona 'volumen': 36416
Unidad de comercializacion
$/kilo (volumen en unidades)                   25807
$/paquete 20 unidades (volumen en unidades)     7106
$/paquete 10 unidades (volumen en unidades)     2066
$/cien unidades (volumen en unidades)           1424
$/cien en rama (volumen en unidades)              13


In [13]:
por_prod = (df.groupby("Producto", observed=True)["Unidad de comercializacion"]
              .nunique().sort_values(ascending=False))
print(por_prod.head(15))
print((por_prod == 1).sum(), "de", len(por_prod), "productos usan una sola unidad")

cuenta = (df.groupby(["Producto", "Unidad de comercializacion"], observed=True)
            .size().rename("n").reset_index())
cuenta["parte"] = cuenta["n"] / cuenta.groupby("Producto", observed=True)["n"].transform("sum")
dominante = cuenta.sort_values("n", ascending=False).drop_duplicates("Producto")
print("\nFilas cubiertas por la unidad dominante de cada producto:",
      round(dominante["n"].sum() / len(df), 3))
print(dominante["parte"].describe().round(3))

Producto
Nectarín     33
Durazno      30
Naranja      24
Damasco      23
Pera         22
Uva          21
Caqui        20
Cereza       20
Mandarina    19
Manzana      17
Palta        16
Limón        16
Kiwi         16
Cebolla      16
Ciruela      15
Name: Unidad de comercializacion, dtype: int64
4 de 82 productos usan una sola unidad

Filas cubiertas por la unidad dominante de cada producto: 0.595
count    82.000
mean      0.651
std       0.241
min       0.218
25%       0.425
50%       0.622
75%       0.875
max       1.000
Name: parte, dtype: float64


In [16]:
candidatos = por_prod[por_prod >= 3].index
mas_filas = (df[df["Producto"].isin(candidatos)]
             .groupby("Producto", observed=True).size().nlargest(4).index)
for p in mas_filas:
    t = (df[df["Producto"] == p]
         .groupby("Unidad de comercializacion", observed=True)
         .agg(filas=("Volumen", "size"), vol_mediano=("Volumen", "median"),
              precio_mediano=("Precio promedio", "median"))
         .query("filas >= 200").sort_values("filas", ascending=False).head(6))
    print("\n", p)
    print(t)


 Manzana
                            filas  vol_mediano  precio_mediano
Unidad de comercializacion                                    
$/bins (400 kilos)          28863         16.0        217500.0
$/caja 16 kilos empedrada   21265        100.0         11000.0
$/bandeja 15 kilos granel   19009        110.0         12000.0
$/caja 18 kilos embalada     8082        150.0         20000.0
$/caja 18 kilos granel       2844        280.0         13000.0
$/caja 15 kilos empedrada    1956        150.0         14000.0

 Lechuga
                            filas  vol_mediano  precio_mediano
Unidad de comercializacion                                    
$/caja 15 unidades          28796        350.0       6500.0000
$/caja 18 unidades          19235        200.0       5000.0000
$/caja 10 unidades          11525        230.0       6000.0000
$/caja 12 unidades           9073        200.0       5000.0000
$/caja 20 unidades           4800        260.0       5272.7272
$/caja 24 unidades            782  

In [14]:
ceros = df[df["Volumen"] == 0]
print(ceros.groupby("anio").size())
print(ceros["Mercado"].value_counts().head(5))
print(ceros["Producto"].value_counts().head(5))

anio
2025    201
dtype: int64
Mercado
Vega Modelo de Temuco                        114
Agrícola del Norte S.A. de Arica              40
Terminal Hortofrutícola Agro Chillán          25
Mercado Mayorista Lo Valledor de Santiago     11
Terminal La Palmera de La Serena              10
Name: count, dtype: int64
Producto
Cereza      51
Manzana     39
Nectarín    32
Durazno     14
Caqui       12
Name: count, dtype: int64


In [15]:
print(df["Origen"].value_counts().to_string())
print()
print(df["Calidad"].value_counts().to_string())
print()
print(sorted(df["Producto"].cat.categories))

Origen
Región de O'Higgins                     244884
Región Metropolitana                    161305
Región de Arica y Parinacota            129493
Provincia de Limarí                     103793
Provincia de Quillota                    99740
Región del Maule                         78804
Provincia del Elquí                      63471
Ecuador                                  60052
Provincia de Melipilla                   58189
Provincia de Curicó                      55038
Perú                                     47041
Región de Coquimbo                       20713
Región de La Araucanía                   17460
China                                    15420
Limache                                  13451
Provincia de Cautín                      12996
Región de Ñuble                          11537
Provincia de San Felipe de Aconcagua     10433
Provincia de Chacabuco                    9692
Cabildo                                   9276
Provincia de Diguillín                    9057
Provin

In [17]:
pd.set_option("display.max_rows", 300)
unidades = (df.groupby("Unidad de comercializacion", observed=True)
              .agg(filas=("Volumen", "size"), productos=("Producto", "nunique"))
              .sort_values("filas", ascending=False))
print(unidades.to_string())

                                              filas  productos
Unidad de comercializacion                                    
$/unidad                                     120522         11
$/bandeja 18 kilos                            83280         11
$/malla 18 kilos                              63372          4
$/bandeja 10 kilos                            57763         21
$/saco 25 kilos                               50674          9
$/bins (400 kilos)                            47902         11
$/caja 15 kilos                               47512         16
$/caja 20 kilos                               44751         15
$/kilo (en caja de 17 kilos)                  41982          1
$/caja 18 kilos                               37247         19
$/bandeja 15 kilos granel                     32557          8
$/caja 16 kilos empedrada                     30080          6
$/caja 15 unidades                            28796          1
$/kilo (volumen en unidades)                  25807    

In [18]:
m = df["Unidad de comercializacion"].astype(str).str.contains("volumen en unidades")
print(df[m].groupby(["Producto", "Unidad de comercializacion"], observed=True)
          .size().rename("n").sort_values(ascending=False).to_string())

Producto  Unidad de comercializacion                 
Zapallo   $/kilo (volumen en unidades)                   23403
Cebolla   $/paquete 20 unidades (volumen en unidades)     6360
Sandia    $/kilo (volumen en unidades)                    2404
Cebolla   $/paquete 10 unidades (volumen en unidades)     2066
Rabanito  $/cien unidades (volumen en unidades)           1424
Ajo       $/paquete 20 unidades (volumen en unidades)      746
Cebolla   $/cien en rama (volumen en unidades)              13


In [22]:
extranjeros = ["Ecuador", "Perú", "China", "Brasil", "EE.UU.", "México", "Bolivia",
               "Argentina", "Colombia", "Costa Rica", "Italia", "Paraguay", "Importada(o)"]
df["origen_tipo"] = np.where(df["Origen"].astype(str).isin(extranjeros),
                             "Extranjero", "Nacional")
df["origen_tipo"] = df["origen_tipo"].astype("category")

print(df["origen_tipo"].value_counts(normalize=True).round(4))
print((df["origen_tipo"].eq("Extranjero").groupby(df["anio"]).mean() * 100).round(2))
print(df[df["origen_tipo"] == "Extranjero"].groupby("Producto", observed=True)
        .size().sort_values(ascending=False).head(15))

origen_tipo
Nacional      0.8981
Extranjero    0.1019
Name: proportion, dtype: float64
anio
2020     8.99
2021     9.74
2022    10.21
2023    10.77
2024    10.83
2025    10.11
2026    10.90
Name: origen_tipo, dtype: float64
Producto
Plátano         32729
Piña            27472
Mango           19430
Palta           17170
Ajo             15302
Limón            2917
Jengibre         2865
Zapallo          2790
Poroto verde     2764
Cebolla          2704
Sandia           2352
Camote           1379
Coco              988
Maracuyá          243
Naranja           233
dtype: int64


In [24]:
from src.unidades import parsear_unidad

etiquetas = pd.Series(df["Unidad de comercializacion"].cat.categories)
mapa = pd.DataFrame(
    [parsear_unidad(e) for e in etiquetas],
    columns=["tipo_unidad", "kilos_unidad"],
)
mapa.insert(0, "etiqueta", etiquetas)

unid = df["Unidad de comercializacion"].astype(object)
df["tipo_unidad"] = unid.map(dict(zip(mapa["etiqueta"], mapa["tipo_unidad"]))).astype("category")
df["kilos_unidad"] = unid.map(dict(zip(mapa["etiqueta"], mapa["kilos_unidad"]))).astype(float)
df["precio_kg"] = df["Precio promedio"] / df["kilos_unidad"]

In [25]:
cob = (df.groupby("tipo_unidad", observed=True)
         .agg(filas=("Volumen", "size"),
              etiquetas=("Unidad de comercializacion", "nunique")))
cob["% filas"] = (cob["filas"] / len(df) * 100).round(1)
print(cob.sort_values("filas", ascending=False))

print()
print(mapa[mapa["tipo_unidad"].isin(["otro", "rango"])].to_string(index=False))

                        filas  etiquetas  % filas
tipo_unidad                                      
kilos_envase           819069        105     63.1
unidades               368148         59     28.4
kilo                    62260         13      4.8
kilo_vol_unidades       25807          1      2.0
rango                   11070          7      0.9
unidades_vol_unidades   10609          4      0.8
otro                      572          2      0.0

                           etiqueta tipo_unidad  kilos_unidad
               $/atado 0,5 a 1 kilo       rango           NaN
              $/atado 1 a 1,5 kilos       rango           NaN
              $/atado 1,5 a 2 kilos       rango           NaN
              $/atado 2,5 a 3 kilos       rango           NaN
           $/atado 300 a 500 gramos       rango           NaN
$/bandeja 12 canastillos 125 gramos        otro           NaN
$/bandeja 12 canastillos 170 gramos        otro           NaN
            $/paquete 1,5 a 2 kilos       rango      

In [26]:
d = df[df["precio_kg"].notna() & (df["Precio promedio"] > 0)]
t = (d.groupby(["Producto", "Unidad de comercializacion"], observed=True)
       .agg(filas=("precio_kg", "size"), pkg_mediano=("precio_kg", "median"))
       .reset_index().query("filas >= 200"))
g = (t.groupby("Producto", observed=True)
       .agg(n_unid=("pkg_mediano", "size"),
            minimo=("pkg_mediano", "min"), maximo=("pkg_mediano", "max")))
g = g[g["n_unid"] >= 2]
g["razon"] = (g["maximo"] / g["minimo"]).round(2)
print(g.sort_values("razon", ascending=False).head(15).round(0))

           n_unid  minimo  maximo  razon
Producto                                
Limón          10   429.0  1583.0    4.0
Uva             9   500.0  1750.0    4.0
Cereza          5   700.0  2400.0    3.0
Tomate          8   375.0  1056.0    3.0
Ají             7   800.0  2150.0    3.0
Ciruela         5   411.0  1107.0    3.0
Naranja        11   400.0   967.0    2.0
Perejil         2  1333.0  3000.0    2.0
Locoto          2  1206.0  2700.0    2.0
Pera            8   500.0  1111.0    2.0
Manzana        10   544.0  1111.0    2.0
Durazno        13   788.0  1550.0    2.0
Mandarina       7   500.0   975.0    2.0
Maracuyá        2  1175.0  2222.0    2.0
Cilantro        2  1412.0  2500.0    2.0


In [27]:
ext = df["origen_tipo"].eq("Extranjero")
res = pd.DataFrame({
    "pct_extranjero": ext.groupby(df["Producto"], observed=True).mean(),
    "filas": df.groupby("Producto", observed=True).size(),
})
siempre = res[res["pct_extranjero"] >= 0.95]
mixtos = res[(res["pct_extranjero"] > 0.01) & (res["pct_extranjero"] < 0.95)]
print("Casi siempre extranjero (>=95%):", list(siempre.index))
print(mixtos.sort_values("filas", ascending=False).round(3))

sub = df[df["Producto"].isin(mixtos.index) & (df["Fecha"].dt.month <= 9)].copy()
sub["ext"] = sub["origen_tipo"].eq("Extranjero")
print((sub.groupby(["Producto", "anio"], observed=True)["ext"].mean()
          .unstack() * 100).round(1))

Casi siempre extranjero (>=95%): ['Camote', 'Coco', 'Jengibre', 'Mango', 'Piña', 'Plátano']
                 pct_extranjero  filas
Producto                              
Palta                     0.235  73134
Limón                     0.041  70876
Cebolla                   0.047  57011
Zapallo                   0.118  23555
Ajo                       0.825  18539
Melón                     0.011  16318
Sandia                    0.173  13587
Poroto verde              0.230  11995
Arándano (blue)           0.019   6572
Poroto granado            0.016   5578
Maracuyá                  0.179   1358
Pera asiática             0.031     97
anio             2020  2021  2022  2023  2024  2025  2026
Producto                                                 
Ajo              64.8  87.7  80.7  89.7  87.7  91.6  85.2
Arándano (blue)   0.0   0.0   3.0   7.8   0.8   4.9   2.7
Cebolla           3.4   6.5   4.4   3.1   4.5   1.9   2.2
Limón             3.4   3.5   3.2   3.1   4.7   5.9   5.9
Maracuyá      